In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import torch
from pathlib import Path
import pprint

# Path to the model
model_path = "/data/jamesliu/models/models--yuhuili--EAGLE3-LLaMA3.1-Instruct-8B/snapshots/607d0d5b7871cd4b89395b6af288c070cfa0a168"

# Check if it's a directory or file
model_path = Path(model_path)

if model_path.is_dir():
    # If it's a directory, look for .bin or .pt files
    model_files = list(model_path.glob("*.bin")) + list(model_path.glob("*.pt"))
    if model_files:
        print(f"Found model files: {[f.name for f in model_files]}")
        # Load the first model file
        model_data = torch.load(model_files[0], map_location="cpu")
    else:
        print(f"No .bin or .pt files found in {model_path}")
        # Try to find a model.safetensors file
        safetensors_file = model_path / "model.safetensors"
        if safetensors_file.exists():
            print(f"Found safetensors file: {safetensors_file}")
            try:
                from safetensors import safe_open
                with safe_open(safetensors_file, framework="pt", device="cpu") as f:
                    tensor_names = f.keys()
                print("Tensor names in safetensors file:")
                pprint.pprint(list(tensor_names))
                # No need to continue with further code
                model_data = {"safetensors_keys": list(tensor_names)}
            except ImportError:
                print("safetensors package not installed. Install with: pip install safetensors")
        else:
            print(f"No model files found in {model_path}")
            model_data = None
else:
    # It's a file, load directly
    model_data = torch.load(model_path, map_location="cpu")

# If model_data is a state_dict (common for models)
if model_data is not None:
    if isinstance(model_data, dict):
        # Get top-level keys
        print("\nTop-level keys:")
        pprint.pprint(list(model_data.keys()))
        
        # For each top-level key, show type and shape (if tensor)
        print("\nStructure details:")
        for key, value in model_data.items():
            if isinstance(value, torch.Tensor):
                print(f"{key}: Tensor with shape {value.shape}, dtype {value.dtype}")
            elif isinstance(value, dict):
                print(f"{key}: Dict with {len(value)} keys")
                # Print a few keys as examples
                if value:
                    sample_keys = list(value.keys())[:3]
                    print(f"  Sample keys: {sample_keys}")
            else:
                print(f"{key}: {type(value)}")

Found model files: ['pytorch_model.bin']


/tmp/ipykernel_1307275/1776671935.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_data = torch.load(model_files[0], map_location="cpu")



Top-level keys:
['d2t',
 't2d',
 'midlayer.self_attn.q_proj.weight',
 'midlayer.self_attn.k_proj.weight',
 'midlayer.self_attn.v_proj.weight',
 'midlayer.self_attn.o_proj.weight',
 'midlayer.mlp.gate_proj.weight',
 'midlayer.mlp.up_proj.weight',
 'midlayer.mlp.down_proj.weight',
 'midlayer.hidden_norm.weight',
 'midlayer.input_layernorm.weight',
 'midlayer.post_attention_layernorm.weight',
 'norm.weight',
 'fc.weight',
 'lm_head.weight']

Structure details:
d2t: Tensor with shape torch.Size([32000]), dtype torch.int64
t2d: Tensor with shape torch.Size([128256]), dtype torch.bool
midlayer.self_attn.q_proj.weight: Tensor with shape torch.Size([4096, 8192]), dtype torch.float16
midlayer.self_attn.k_proj.weight: Tensor with shape torch.Size([1024, 8192]), dtype torch.float16
midlayer.self_attn.v_proj.weight: Tensor with shape torch.Size([1024, 8192]), dtype torch.float16
midlayer.self_attn.o_proj.weight: Tensor with shape torch.Size([4096, 4096]), dtype torch.float16
midlayer.mlp.gate_pro

In [1]:
print(model_data["model"])

NameError: name 'model_data' is not defined